In [37]:
fecha_mes_base='2026-08-01'
tipi_cond1='SAE'
tipi_cond2='SAR'
tipi_cond3='AGENDA'
tb_tipolofia='tTipologia_Cencosud_PPFF'
servidor_01=76
tipi_cod='Codigo'
tipi_resp_cod='R0'
tipi_descrip='DESCRIPCION'
tipi_estado='tipo'
tipi_resp_estado='NO GESTIONADO'
tipi_subdescripcion='SUB_DESCRIPCION'
tnum_tb='tNumeroCenco_Sae'
tnum_dni='CODDOC'
tlista_generada='borrar_prestamo_cencosud'
get_base=since_base_maestra_cencosud_ppff

def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
    query = f"""
        SELECT *
        FROM OPENQUERY([192.168.3.{servidor_01}], '
            SELECT        
            rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
            e.dial_method,
            a.campaign_id AS numero_campana,        
            a.user AS dni_ejecutivo,
            c.full_name AS ejecutivo,
            e.campaign_name AS nombre_campana,        
            a.call_date AS fecha_hora_llamada,        
            a.length_in_sec AS duracion,        
            b.status_name AS call_result,        
            f.list_description,        
            f.list_name,        
            a.phone_number as phone_number,        
            d.alt_phone as fecha_agenda,        
            d.comments as comentarios,        
            a.status AS codigo,
            a.term_reason,	
            a.alt_dial
            FROM asterisk.vicidial_log a         
            LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
            LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
            LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
            LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
            LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
            WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
            AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
            AND a.call_date < 
            DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
        ')

        """
    df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso  FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','term_reason','alt_dial','call_result','duracion','codigo','nombre_campana')

df_vici=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)
df_vici.filter(F.col('vendor_lead_code')=='25666697').orderBy(F.col('fecha_hora_llamada').desc()).show(truncate=False)


+-------------------+----------------------------+----------------+------------+--------------------------------+-------------+-----------------------------+-----------------------+------------+--------+----------------------+--------+------+--------------+
|fecha_hora_llamada |list_name                   |vendor_lead_code|phone_number|descripcion                     |dni_ejecutivo|ejecutivo                    |dial_method            |term_reason |alt_dial|call_result           |duracion|codigo|nombre_campana|
+-------------------+----------------------------+----------------+------------+--------------------------------+-------------+-----------------------------+-----------------------+------------+--------+----------------------+--------+------+--------------+
|2026-08-07 17:03:25|MEDIO/MAYORA10K             |25666697        |998067641   |CLIENTE ACEPTA PRODUCTO         |PFC111       |REN ROBERTO LEYZAQUIA RAMOS  |RATIO                  |AGENT       |MAIN    |NULL                  |











lista 





feedback


envio formulario
correo


In [ ]:
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	select NUMERO_DOCUMENTO
	from DANTALION.dbo.Base_Maestra_Diners_TC_Vigente
    WHERE RETIRO IS NULL OR RETIRO =''
"""
df_tc = pd.read_sql(query, engine_kishin)

In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [25]:
import os
import re
import glob
import pandas as pd

lista_df = []

for archivo in glob.glob(os.path.join(ruta_alfin, "FEEDBACK_TARGET_*.csv")):

    df_temp = cargar_archivo_csv_ruta(spark,archivo,';',True,ruta_alfin)

    nombre_archivo = os.path.basename(archivo)

    df_temp = df_temp.withColumn(
        "archivo",
        F.lit(nombre_archivo)
    )

    lista_df.append(df_temp)

In [28]:
from functools import reduce
df_feedback = reduce(
    lambda df1, df2: df1.unionByName(
        df2,
        allowMissingColumns=True
    ),
    lista_df
)

AttributeError: 'DataFrame' object has no attribute 'unionByName'

In [29]:
import os
import re
import glob
import pandas as pd

lista_df = []

for archivo in glob.glob(os.path.join(ruta_alfin, "FEEDBACK_TARGET_*.csv")):

    df_temp = pd.read_csv(archivo,sep='|')

    nombre = os.path.basename(archivo)

    fecha = re.search(r"(\d{8})", nombre).group(1)

    df_temp["fecha_archivo"] = pd.to_datetime(
        fecha,
        format="%Y%m%d"
    )

    lista_df.append(df_temp)



In [30]:
df.head()

,COD_CANAL,CANAL,DNI,FECHA_ENVIO,FECHA_GESTION,HORA_GESTION,TELEFONO,ORIGEN_TELEFONO,COD_CAMPANA,COD_TIPO,OFERTA,DNI_ASESOR,fecha_archivo
0,TAG202,TARGET,489055,20260801,20260801,10:18:58,952384702,T1,SDFCP100,AB,12000,NaN,2026-08-01
1,TAG202,TARGET,493223,20260801,20260801,14:02:29,982704526,T1,MUJER,NaN,25000,NaN,2026-08-01
2,TAG202,TARGET,493867,20260801,20260801,17:05:51,952515977,T1,SOLODNI,PDROP,2700,NaN,2026-08-01
3,TAG202,TARGET,2291394,20260801,20260801,13:54:29,906190174,T1,SOLODNI,AB,2700,NaN,2026-08-01
4,TAG202,TARGET,2295039,20260801,20260801,17:37:20,944010441,T1,MUJER,PDROP,5000,NaN,2026-08-01


In [31]:
df.groupby("FECHA_GESTION").size().reset_index(name="cantidad")

,FECHA_GESTION,cantidad
0,20260801,8690
1,20260802,3964
2,20260803,7869
3,20260804,9092
4,20260805,6834
5,20260806,5729


In [34]:
df_1=df.drop_duplicates('DNI').copy()

In [35]:
df_1.groupby("FECHA_GESTION").size().reset_index(name="cantidad")


,FECHA_GESTION,cantidad
0,20260801,7204
1,20260802,3854
2,20260803,5180
3,20260804,6887
4,20260805,6241
5,20260806,5729


In [36]:
df_1.shape

(35095, 13)

In [23]:
df.shape

(42178, 13)

In [18]:
df = pd.concat(
    lista_df,
    ignore_index=True
)

In [9]:
df.count()

COD_CANAL          42178
CANAL              42178
DNI                42178
FECHA_ENVIO        42178
FECHA_GESTION      42178
HORA_GESTION       42178
TELEFONO           42178
ORIGEN_TELEFONO    42178
COD_CAMPANA        42178
COD_TIPO           34335
OFERTA             42178
DNI_ASESOR             0
fecha_archivo      42178
dtype: int64

In [10]:
query = """
select 
'TAG202' as COD_CANAL,
'TARGET' as CANAL,
a.Dni as DNI,
'20260801' as FECHA_ENVIO,
FORMAT(a.Fecha_Llamada, 'yyyyMMdd') AS FECHA_GESTION,
CONVERT(VARCHAR(8), a.Fecha_Hora_Llamada, 108) AS HORA_GESTION,
a.PHONE_NUMBER as TELEFONO,
'T1' as ORIGEN_TELEFONO,
b.campania as COD_CAMPANA,
case 
    WHEN a.DNI_Ejecutivo = 'VDAD' THEN '27' 
    WHEN a.DNI_Ejecutivo = 'XFER' THEN '27' 
    WHEN a.DNI_Ejecutivo = 'DCMX' THEN '27' 
    WHEN a.Codigo_paleta = 'DISPO' THEN '27' 
    WHEN a.Codigo_paleta = 'DCMX' THEN '27' 
    WHEN a.Codigo_paleta = 'XFER' THEN '27' 
    WHEN a.Codigo_paleta = 'INCALL' THEN '27' 
    WHEN a.Codigo_paleta = 'DONEM' THEN '27' 
    WHEN a.Codigo_paleta = '32' THEN '24' 
    else a.Codigo_paleta 
end as COD_TIPO,
b.oferta_max as OFERTA,
'' as DNI_ASESOR
,fecha_llamada
from THOTH.dbo.Tmp_LLamadas_Alfin a
inner join ODIN.dbo.Base_Maestra_Alfin_bk_Vigente b
on a.Dni=b.NUMERO_DOCUMENTO
where b.NUMERO_DOCUMENTO<>'10492705'
    """
df_formato=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_formato.columns)


['COD_CANAL', 'CANAL', 'DNI', 'FECHA_ENVIO', 'FECHA_GESTION', 'HORA_GESTION', 'TELEFONO', 'ORIGEN_TELEFONO', 'COD_CAMPANA', 'COD_TIPO', 'OFERTA', 'DNI_ASESOR', 'fecha_llamada']


In [3]:
df_formato.select('FECHA_GESTION').orderBy(F.col('FECHA_GESTION').desc()).show()

+-------------+
|FECHA_GESTION|
+-------------+
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
|     20260806|
+-------------+
only showing top 20 rows


In [ ]:
df_formato

12214

In [11]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

# dni_retiro = set(df_retiros['dni_cliente'].dropna())
# cel_retiro = set(df_retiros['celular'].dropna())

# df_seguimiento_1['retiro'] = (
#     df_seguimiento_1['dni_cliente'].isin(dni_retiro) |
#     df_seguimiento_1['celular'].isin(cel_retiro)
# ).astype(int)

C:\Users\DATA\AppData\Local\Temp\ipykernel_7220\2298311491.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [12]:
import os
import pandas as pd

# Convertir el DataFrame de PySpark a Pandas
df_feedback_pd = df_formato.toPandas()
# df_feedback_pd.head()

In [13]:
df_feedback_pd.shape


(42282, 13)

In [14]:
# df_validar = df_postulante[
#     ~df_postulante['id_postulante'].isin(df_retiros['dni_cliente'])
# ].copy()

# df_embudo['numero_de_documento'] = (
#     df_embudo['numero_de_documento']
#     .astype(str)
#     .str.replace(r'\D', '', regex=True)      # deja solo números
#     .replace('', pd.NA)                      # vacío -> NA
#     .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
#     .str.zfill(8)                            # <8 dígitos -> completa con ceros
# )

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

mask = (
    df_feedback_pd['DNI'].isin(dni_retiro) |
    df_feedback_pd['TELEFONO'].isin(cel_retiro)
)

df_feedback_pd = df_feedback_pd[~mask]

df_feedback_pd.shape



(42281, 13)

In [15]:

# Convertir la columna de fecha
df_feedback_pd["FECHA_INICIO_GESTION_ref"] = pd.to_datetime(
    df_feedback_pd["fecha_llamada"],
    errors="coerce"
)

# Crear una columna auxiliar solo con la fecha
df_feedback_pd["fecha_archivo"] = (
    df_feedback_pd["FECHA_INICIO_GESTION_ref"].dt.date
)

# Eliminar registros cuya fecha no pudo convertirse
df_feedback_pd = df_feedback_pd[
    df_feedback_pd["fecha_archivo"].notna()
]

# Crear la carpeta si no existe
os.makedirs(ruta_csv, exist_ok=True)

# Generar un archivo CSV por cada fecha
for fecha, df_dia in df_feedback_pd.groupby("fecha_archivo"):
    nombre_fecha = fecha.strftime("%Y%m%d")
    nombre_archivo = f"FEEDBACK_TARGET_{nombre_fecha}.csv"
    # nombre_fecha = fecha.strftime("%d_%m_%Y")
    # nombre_archivo = f"FEEDBACK_TARGET_{nombre_fecha}.csv"

    ruta_archivo = os.path.join(
        ruta_csv,
        nombre_archivo
    )

    # Quitar columnas auxiliares antes de exportar
    df_exportar = df_dia.drop(
        columns=[
            "FECHA_INICIO_GESTION_ref",
            "fecha_archivo",
            'fecha_llamada'
        ]
    )

    # Exportar CSV separado por |
    df_exportar.to_csv(
        ruta_archivo,
        sep="|",
        index=False,
        encoding="utf-8-sig",
        lineterminator="\n"
    )

    print(
        f"Archivo generado: {ruta_archivo} "
        f"| filas: {len(df_exportar)}"
    )

Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260801.csv | filas: 8690
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260802.csv | filas: 3964
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260803.csv | filas: 7869
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260804.csv | filas: 9092
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260805.csv | filas: 6834
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260806.csv | filas: 5729
Archivo generado: C:\Users\DATA\Documents\datos\05_subir_csv\FEEDBACK_TARGET_20260807.csv | filas: 103


In [ ]:
FEEDBACK_TARGET_20260702

In [140]:
df_formato.show()

+---------+------+--------+-----------+-------------+------------+---------+---------------+-----------+--------+------+----------+-------------+
|COD_CANAL| CANAL|     DNI|FECHA_ENVIO|FECHA_GESTION|HORA_GESTION| TELEFONO|ORIGEN_TELEFONO|COD_CAMPANA|COD_TIPO|OFERTA|DNI_ASESOR|fecha_llamada|
+---------+------+--------+-----------+-------------+------------+---------+---------------+-----------+--------+------+----------+-------------+
|   TAG202|TARGET|00219827|   20260701|     20260704|    04:33:06|976149271|             T1|    SOLODNI|      27| 18000|          |   2026-07-04|
|   TAG202|TARGET|02837980|   20260701|     20260701|    09:18:55|968913228|             T1|      MUJER|      27|  2200|          |   2026-07-01|
|   TAG202|TARGET|02838590|   20260701|     20260701|    05:42:49|966575545|             T1|    SOLODNI|      27|  1900|          |   2026-07-01|
|   TAG202|TARGET|06555251|   20260701|     20260704|    11:16:49|993904077|             T1|      SD PE|      27|  6200|    

In [ ]:
df = df.withColumn(
    "hora",
    F.date_format(F.col("fecha_hora_llamada"), "HH:mm:ss")
)

In [109]:
df_formato.show()

+--------+--------------+-----------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+-----------+-------------+-------------------+-------------------+--------+--------+--------------------+--------------------+-----+------------------+----------+------------+---+-------+--------------+---------------+----+
|     Dni|Numero_Campana|   Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|         Descripcion|list_description|       list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin| lead_id| Estado_|         Sub_Estado_|        Descripcion_|Pesos|            Enlace|Fecha_Llam|Hora_Llamada| RH|COD_BCO|Mejor_Telefono|Mejor_Resultado|RHFC|
+--------+--------------+-----------------+-------------+------------------+----------------

In [ ]:
['Dni', 'Numero_Campana', 'Nombre_Campana', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'Estados', 'Sub_estado', 'Descripcion', 'list_description', 'list_name', 'PHONE_NUMBER', 'Fecha_Agenda', 'Comentarios', 'Codigo_Paleta', 'Inicio', 'Fin', 'lead_id', 'Estado_', 'Sub_Estado_', 'Descripcion_', 'Pesos', 'Enlace', 'Fecha_Llam', 'Hora_Llamada', 'RH', 'COD_BCO', 'Mejor_Telefono', 'Mejor_Resultado', 'RHFC']celular

In [ ]:


'
SELECT 
    RTRIM(LTRIM(d.vendor_lead_code)) AS dni_cliente,
    DATE(a.call_date) AS fecha,
    TIME(a.call_date) AS hora,
    a.PHONE_NUMBER AS celular,
    CASE 
        WHEN DNI_Ejecutivo = ''VDAD'' THEN 27 
        ELSE a.status 
    END AS tipificacion,
    ''17'' AS servicio,
    ''30'' AS id_supervisor,
    '''' AS campana,
    ''1'' AS intentos,
    a.length_in_sec AS TMO,
    ''0'' AS T_espera,
    ''TARGET'' AS ID_RECORD,
    CASE 
        WHEN a.user = ''VDAD'' THEN ''99999999'' 
        ELSE ''00000001'' 
    END AS edni,
    CASE 
        WHEN a.user <> ''VDAD'' THEN ''00000001'' 
        ELSE a.user 
    END AS eid
FROM asterisk.vicidial_log a 
LEFT JOIN asterisk.vicidial_list d 
    ON a.lead_id = d.lead_id
LEFT JOIN asterisk.vicidial_campaigns e 
    ON a.campaign_id = e.campaign_id
LEFT JOIN asterisk.vicidial_lists f 
    ON a.list_id = f.list_id
LEFT JOIN asterisk.vicidial_statuses b 
    ON a.status = b.status
LEFT JOIN asterisk.vicidial_users c 
    ON a.user = c.user
WHERE DATE(a.call_date) = ''2026-07-01''
  AND a.campaign_id = ''401''
'

In [107]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

    

## lista seguimiento

In [78]:
ruta_archivo = os.path.join(ruta_csv, 'ULTIMO_.csv')
df_seguimiento=cargar_archivo_csv(spark,ruta_archivo,';',True)
print(df_seguimiento.columns)


query = """
    select NUMERO_DOCUMENTO as DNI2
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
print(df_formato.columns)

['DNI0', 'D', 'DNI2', 'NOMBRE', 'CELULAR', 'COD', 'AGENCIA', '_c7', 'MONTO']
['DNI2']


In [79]:
df_seguimiento = df_seguimiento.withColumn(
    "DNI2",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI2")),
        F.lit(8)
    )
)
df_formato = df_formato.withColumn(
    "DNI2",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI2")),
        F.lit(8)
    )
)

In [81]:
append_table_SQL(spark,df_formato,f'ventas_alfin_a',server_zeus,user_zeus,pwd_zeus,'odin')


In [80]:
df_formato=df_formato.join(df_seguimiento,['DNI2'],'inner')

In [70]:
df_formato.show()

+--------+--------+----------------+--------------+----------------+-------------------+--------+---------------+------------+--------------+------------+--------------+----------+-----------+-----------------+------------+---------+----------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+---------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+---------------+-------------+------

In [4]:


# df_seguimiento = df_seguimiento.withColumn(
#     "DNI",
#     F.date_format(
#         F.to_date(F.col("DNI"), "d/MM/yyyy"),
#         "yyyy-MM-dd"
#     )
# )
df_seguimiento = df_seguimiento.withColumn(
    "DNI",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
)
df_seguimiento=df_seguimiento.join(df_formato,['DNI'],'left')

df_seguimiento=df_seguimiento.withColumn('CLIENTE',when(F.col('CLIENTE').isNull(),F.col('NOMBRES')).otherwise(F.col('CLIENTE')))
df_seguimiento=df_seguimiento.withColumnRenamed('DNI','vendor_lead_code')
df_seguimiento=df_seguimiento.withColumnRenamed('CLIENTE','address1')
df_seguimiento=df_seguimiento.withColumnRenamed('TELEFONO','phone_number')
df_seguimiento=df_seguimiento.withColumnRenamed('PLAZA','email')
df_seguimiento=df_seguimiento.withColumnRenamed('MONTO','comments')
df_seguimiento=df_seguimiento.select('vendor_lead_code','address1','phone_number','email','comments')

df_list_filtrada = df_seguimiento.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin_seguimiento.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)


## lista BOT

In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [34]:
# filename='BASE_TARGET_20260701_alfin.csv'
# df_base=cargar_archivo_csv(spark,filename,';',True)

filename='ULTIMO.csv'
agendas_01=cargar_archivo_csv(spark,filename,';',True)
agendas_01.columns  
filename='lista_01.csv'
df_validar_01=cargar_archivo_csv(spark,filename,';',True)
filename='lista_02.csv'
df_validar_02=cargar_archivo_csv(spark,filename,';',True)
# filename='LISTA_ALFIN_1.csv'
# df_validar_03=cargar_archivo_csv(spark,filename,';',True)
# filename='LISTA_ALFIN_2.csv'
# df_validar_04=cargar_archivo_csv(spark,filename,';',True)

df_validar=df_validar_01.unionByName(df_validar_02)

print(agendas_01.columns)
print(df_validar.columns)

['DNI', 'NOMBRES', 'CELULAR', 'COD', 'AGENCIA', 'FECHA', 'MONTO']
['DNI', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL']


In [35]:
agendas_01=agendas_01.join(df_validar,['DNI'],'inner')


In [28]:
agendas_01.join(df_validar,['DNI'],'inner').count()

2469

In [36]:
# agendas_01=agendas_01.filter(F.col('PROPENSION_DISTRIBUCION').isin('1','2'))
agendas_01.groupBy('PROPENSION_DISTRIBUCION') \
    .count() \
    .orderBy('PROPENSION_DISTRIBUCION') \
    .show(30)


+-----------------------+-----+
|PROPENSION_DISTRIBUCION|count|
+-----------------------+-----+
|                      1| 1135|
|                      2| 1046|
|                      3|   55|
|                      4|   50|
|                      5|   70|
|                      6|  113|
+-----------------------+-----+



In [37]:
agendas_01=agendas_01.filter(F.col('PROPENSION_DISTRIBUCION').isin('1'))

In [32]:
agendas_01 = agendas_01.withColumn(
    "rango_OFERTA",   
    F.when(F.col("OFERTA_MAX").cast("int").isNull(), "SIN DATO")
     .when(F.col("OFERTA_MAX").cast("int") < 2500,  "00.[0 - 2,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 5000,  "01.[2,500 - 5,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 7500,  "02.[5,000 - 7,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 10000, "03.[7,500 - 10,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 12500, "04.[10,000 - 12,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 15000, "05.[12,500 - 15,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 17500, "06.[15,000 - 17,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 20000, "07.[17,500 - 20,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 22500, "08.[20,000 - 22,500)")
     .when(F.col("OFERTA_MAX").cast("int") < 25000, "09.[22,500 - 25,000)")
     .when(F.col("OFERTA_MAX").cast("int") < 27500, "10.[25,000 - 27,500)")
     .otherwise("11.[27,500 A MÁS]")
)

In [33]:
agendas_01=agendas_01.filter(F.col('OFERTA_MAX')>5000)
agendas_01.groupBy('RANGO_OFERTA') \
    .count() \
    .orderBy('RANGO_OFERTA') \
    .show(30)


+--------------------+-----+
|        RANGO_OFERTA|count|
+--------------------+-----+
|  02.[5,000 - 7,500)|  158|
| 03.[7,500 - 10,000)|   92|
|04.[10,000 - 12,500)|  103|
|05.[12,500 - 15,000)|   73|
|06.[15,000 - 17,500)|   43|
|07.[17,500 - 20,000)|   30|
|08.[20,000 - 22,500)|   21|
|09.[22,500 - 25,000)|   10|
|10.[25,000 - 27,500)|    3|
+--------------------+-----+



In [7]:
agendas_01=agendas_01.filter(F.col('FRESCURA').isin('1','4'))

In [4]:
# query = """
#     select *
#     from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
#     where (cl_telf1 is not null and cl_telf1<>0) 
#     and cl_base='julio 2026'
#     and OFERTA_MAX>=5000
#     """
# df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


query = """
    select NUMERO_DOCUMENTO,cl_telf1 as [PHONE NUMBER],NOMBRES,APELLIDO_PATERNO,APELLIDO_MATERNO
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where (cl_telf1 is not null and cl_telf1<>0) 
    and cl_base='julio 2026'
    and OFERTA_MAX>=5000
    """
df_dni_telf=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
query = """
    select *
    from maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    """
df_maestra=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

df_formato=df_maestra.join(df_dni_telf,['NUMERO_DOCUMENTO'],'inner')

In [5]:


df_formato=df_formato.withColumnRenamed('NUMERO_DOCUMENTO','VENDOR LEAD CODE')
# df_formato=df_formato.withColumnRenamed('cl_telf1','PHONE NUMBER')
df_formato=df_formato.withColumnRenamed('Agencia_comercial','TIENDA_CERCANA_LIMPIA')
df_formato=df_formato.withColumn('campana',F.lit('401'))
df_formato=df_formato.withColumnRenamed('COLOR_FINAL','TIENDA_2_DIRECCION')
df_formato=df_formato.withColumnRenamed('OFERTA_MAX','OFERTA_MAX_TEXTO')
df_formato=df_formato.withColumnRenamed('MARCA1','AGENCIA')
df_formato = df_formato.withColumn(
    "ADDRESS1",
    F.concat_ws(
        " ",
        F.col("NOMBRES"),
        F.col("APELLIDO_PATERNO"),
        F.col("APELLIDO_MATERNO")
    )
)
print(df_formato.columns)
# print(agendas_01.columns)


['VENDOR LEAD CODE', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'tMontoDesemFugas', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'TIENDA_CERCANA_LIMPIA', 'Region_comercial', 'TIENDA_2_DIRECCION',

In [ ]:
START TRANSACTION;

-- =========================================================================
-- PASO 1: ALIMENTAR VICIDIAL_LIST (Mapeo Nativo de 14 Campos Operativos)
-- =========================================================================
INSERT INTO asterisk.vicidial_list 
    (
        phone_number, vendor_lead_code, list_id, status, gmt_offset_now, 
        title, first_name, last_name, address1, address2, address3, 
        city, province, email, security_phrase, comments, entry_date, called_since_last_reset
    )
SELECT 
    phone_num, vendor_code, list_id, 'NEW', '0.00', 
    title, first_name, last_name, add1, add2, add3, 
    city, prov, email, sec_phrase, comms, NOW(), 'N'
FROM asterisk.tmp_carga_manual_bot
WHERE procesado = 'N';

-- =========================================================================
-- PASO 2: ALIMENTAR COLLABORIX_LEADS (Cruce Relacional Automatizado)
-- =========================================================================
INSERT INTO asterisk.collaborix_leads 
    (
        lead_id, dni, nombre, nombre_corto, campana, 
        oferta_max_texto, direccion_limpia, tienda_cercana_limpia, 
        tienda_2_nombre, tienda_2_direccion, tienda_3_nombre, tienda_3_direccion, 
        tratamiento, fecha_carga, codAgencia, perfil
    )
SELECT 
    v.lead_id, 
    t.vendor_code, 
    t.add1, -- Contiene el Nombre Completo original
    -- Extrae la primera palabra antes del primer espacio de forma limpia
    SUBSTRING_INDEX(t.add1, ' ', 1),
    '401', -- ID de Campaña por defecto para el Flujo Bot
    t.oferta_max, 
    t.dir_limpia, 
    t.tienda_cercana, 
    t.tienda_2_nom, 
    t.tienda_2_dir, 
    t.tienda_3_nom, 
    t.tienda_3_dir, 
    t.tratamiento, 
    NOW(), 
    t.agencia, 
    'NUEVO'
FROM asterisk.vicidial_list v
INNER JOIN asterisk.tmp_carga_manual_bot t 
    ON v.vendor_lead_code = t.vendor_code AND v.list_id = t.list_id
WHERE t.procesado = 'N'
ON DUPLICATE KEY UPDATE dni = VALUES(dni);

-- =========================================================================
-- PASO 3: CIERRE DE COLA MANUAL
-- =========================================================================
UPDATE asterisk.tmp_carga_manual_bot 
SET procesado = 'Y' 
WHERE procesado = 'N';

COMMIT;

In [ ]:
['VENDOR LEAD CODE', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'tMontoDesemFugas', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'TIENDA_CERCANA_LIMPIA', 'Region_comercial', 'TIENDA_2_DIRECCION', 'GRUPO_TASA', 'GRUPO_MONTO', 'tipo_cliente_riegos', 'USER_V3', 'TIPO_CLIENTE', 'FRESCURA', 'TIPO_BASE', 'campania', 'FLG_AAHH', 'INTENSIDAD_MAX', 'lote', 'REGION', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_TASA', 'TIPO_LOTE', 'TIPO_TELF', 'PHONE NUMBER', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'campana', 'ADDRESS1']per


In [ ]:
# grupo_tasa=['None', '', '', '', '', ]
# agencia_comercial=[ 'SAN MIGUEL', 'MIRAFLORES']

df_filtrado = df_formato.filter(
    # (F.col('propension_ic').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    (F.col('RECORRIDO').isin('0'))&
    # (
    (F.col('PROPENSION_IC').isin('1','2'))&
    (F.col('frescura').isin('4'))&
    # )
    (F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','3. MES + PLD Peers'))&
    (F.col('TIPO_CLIENTE').isin('INDEPENDIENTE'))&
    (F.col('RETIRO')=='ACTIVO')  # retiro

)



print(df_filtrado.count())
print(df_filtrado.columns)

9722
['VENDOR LEAD CODE', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'tMontoDesemFugas', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'TIENDA_CERCANA_LIMPIA', 'Region_comercial', 'TIENDA_2_DIRECC

In [ ]:
['VENDOR LEAD CODE', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'tMontoDesemFugas', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'TIENDA_CERCANA_LIMPIA', 'Region_comercial', 'TIENDA_2_DIRECCION', 'GRUPO_TASA', 'GRUPO_MONTO', 'tipo_cliente_riegos', 'USER_V3', 'TIPO_CLIENTE', 'FRESCURA', 'TIPO_BASE', 'campania', 'FLG_AAHH', 'INTENSIDAD_MAX', 'lote', 'REGION', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_TASA', 'TIPO_LOTE', 'TIPO_TELF', 'PHONE NUMBER', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'campana', 'ADDRESS1']recorri


In [ ]:
df_formato.groupBy('region_comercial','DETALLE_TASA') \
    .count() \
    .orderBy('region_comercial','DETALLE_TASA') \
    .show(30)


In [15]:
engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)
df_agencia.head()

,agencia_Formulario,agencia_base,agencia_base2,agencia_correo,correos
0,738363 - CAJAMARCA,CAJAMARCA,CAJAMARCA,CAJAMARCA,claudia.cantera@alfinbanco.pe
1,737490 - CASTILLA,CASTILLA,CASTILLA,CASTILLA,randy.carrion@alfinbanco.pe
2,734281 - CHICLAYO BALTA,CHICLAYO BALTA,CHICLAYO BALTA,CHICLAYO BALTA,karina.ruiz@alfinbanco.pe
3,734272 - CHIMBOTE,CHIMBOTE,CHIMBOTE,CHIMBOTE,junior.sanchez@alfinbanco.pe
4,738360 - MOSHOQUEQUE,MOSHOQUEQUE,MOSHOQUEQUE,MOSHOQUEQUE,juan.arbulu@alfinbanco.pe


In [17]:
df_formato.show(5)

+----------------+---+--------+------------+--------------------+--------------------+----------------+--------------------+--------------------+----------------+--------------------+--------------------+---------+---------+-------------+-------------+---+-------+-------+--------+------------+------------+------------+--------------+---------+-------------+---+------------------+----+-----------+--------+---------+-------------+-------------+--------+--------+--------+--------+---------+-------+------+---------+---------+-------+-------------+----------+-----------+----------------+------------+----------------+----------------+-----------------+-----------------+-----------+--------+------+-----+-----+-----------+-----------+--------------------+------------------+-------------------+------------+-----------+-------------+-------+---------------------+--------------------+------------------+-------------+---------------+-------------------+--------------------+-------------+--------+-

In [ ]:
    user_bot = "kishin"
pwd_bot = ")k9t-NT5UcMM5iPn"
server_bot = "192.168.3.7"
db_bot = "asterisk"


In [14]:
df_filtrado_1=df_filtrado.select('VENDOR LEAD CODE','PHONE NUMBER','TIENDA_CERCANA_LIMPIA','AGENCIA','TIENDA_2_DIRECCION','OFERTA_MAX_TEXTO','ADDRESS1','campana')
df_filtrado_1.show()

{"ts": "2026-07-10 12:18:33.437", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `AGENCIA` cannot be resolved. Did you mean one of the following? [`CET`, `DIA`, `GESTION`, `NRG`, `REP1`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor21.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o308.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `AGENCIA` cannot be resolved. Did you mean one of the following? [`CET`, `DIA`, `GESTION`, `NRG`, `REP1`]. SQLSTATE: 42703;\n'Project [VENDOR LEAD CODE#564, PHONE NUMBER#473, TIENDA_CERCANA_LIMPIA#565, 'AGENCIA, TIENDA_2_DIRECCION#567, 'OFERTA_MAX_TEXTO, ADDRESS1#568, campana#566]\n+- 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `AGENCIA` cannot be resolved. Did you mean one of the following? [`CET`, `DIA`, `GESTION`, `NRG`, `REP1`]. SQLSTATE: 42703;
'Project [VENDOR LEAD CODE#564, PHONE NUMBER#473, TIENDA_CERCANA_LIMPIA#565, 'AGENCIA, TIENDA_2_DIRECCION#567, 'OFERTA_MAX_TEXTO, ADDRESS1#568, campana#566]
+- Filter ((((PROPENSION_IC#542 IN (1,2) AND frescura#552 IN (4)) AND USER_V3#550 IN (7. Peers,4. MES + PLD No Peers,3. MES + PLD Peers)) AND TIPO_CLIENTE#551 IN (INDEPENDIENTE)) AND (RETIRO#532 = ACTIVO))
   +- Project [VENDOR LEAD CODE#564, NRG#478, Dni#479, TIPO_GESTION#480, GESTION#481, SUBGESTION#482, TIPO_GESTION_hum#483, GESTION_hum#484, SUBGESTION_hum#485, TIPO_GESTION_bot#486, GESTION_bot#487, SUBGESTION_bot#488, TIPO#489, RECORRIDO#490, RECORRIDO_hum#491, RECORRIDO_bot#492, CET#493, CET_hum#494, CET_bot#495, CONT_GEN#496, CONT_GEN_hum#497, CONT_GEN_bot#498, Hora_Llamada#499, Mejor_Telefono#500L,

In [ ]:
['DNI', 'NOMBRE', 'CELCULAR', '#¡VALOR!', 'MONTO', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'rango_OFERTA']CO

['DNI', 'NOMBRE', 'CELCULAR', '#¡VALOR!', 'MONTO', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'rango_OFERTA']


In [62]:
agendas_01.show()

+----------------+--------------------+------------+---------------------+-----+------------------+-----------+--------------------+-------------+-----------+----------+-----+-------------+--------+-----------+------------+----------------+---------------------+------+------+------+------+------+------+------+-----+--------+------------------+-------------------+-------------+----------------+-----------------------+---------+---------+---------+---------+---------+---------+---------+---------+--------------+----+---------------+--------------------+-------+
|VENDOR LEAD CODE|            ADDRESS1|PHONE NUMBER|TIENDA_CERCANA_LIMPIA|MONTO|TIENDA_2_DIRECCION|COD_USER_V3|             USER_V3|    PERFIL_RO|    campaña|OFERTA_MAX|PLAZO|CAPACIDAD_MAX|FRESCURA|rango_deuda|numentidades|TOTAL_A_LIQUIDAR|TASA_CREDITO_ANTERIOR|TASA_1|TASA_2|TASA_3|TASA_4|TASA_5|TASA_6|TASA_7|MGNEG|MARCA_PD|AUTORIZACION_DATOS|FLAG_DEUDA_V_OFERTA|   GRUPO_TASA|       TIPO_BASE|PROPENSION_DISTRIBUCION|OFERTA_SS|TASA

In [13]:
print(agendas_01.columns)
print(df_filtrado_1.columns)

['DNI', 'NOMBRE', 'CELCULAR', '#¡VALOR!', 'MONTO', 'COLOR_FINAL', 'COD_USER_V3', 'USER_V3', 'PERFIL_RO', 'campaña', 'OFERTA_MAX', 'PLAZO', 'CAPACIDAD_MAX', 'FRESCURA', 'rango_deuda', 'numentidades', 'TOTAL_A_LIQUIDAR', 'TASA_CREDITO_ANTERIOR', 'TASA_1', 'TASA_2', 'TASA_3', 'TASA_4', 'TASA_5', 'TASA_6', 'TASA_7', 'MGNEG', 'MARCA_PD', 'AUTORIZACION_DATOS', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'TIPO_BASE', 'PROPENSION_DISTRIBUCION', 'OFERTA_SS', 'TASA_1_SS', 'TASA_2_SS', 'TASA_3_SS', 'TASA_4_SS', 'TASA_5_SS', 'TASA_6_SS', 'TASA_7_SS', 'ALERTA_MAQUETA', 'FEN', 'PERFIL_ESPECIAL', 'rango_OFERTA']
['VENDOR LEAD CODE', 'PHONE NUMBER', 'TIENDA_CERCANA_LIMPIA', 'AGENCIA', 'TIENDA_2_DIRECCION', 'OFERTA_MAX_TEXTO', 'ADDRESS1', 'campana']


In [17]:
agendas_01=agendas_01.select('VENDOR LEAD CODE','PHONE NUMBER','TIENDA_CERCANA_LIMPIA','AGENCIA','TIENDA_2_DIRECCION','OFERTA_MAX_TEXTO','ADDRESS1','campana')


In [14]:
agendas_01=agendas_01.withColumn('campana',F.lit('401'))
agendas_01=agendas_01.withColumnRenamed('DNI','VENDOR LEAD CODE')
agendas_01=agendas_01.withColumnRenamed('NOMBRE','ADDRESS1')
agendas_01=agendas_01.withColumnRenamed('COLOR_FINAL','TIENDA_2_DIRECCION')

agendas_01=agendas_01.withColumnRenamed('CELCULAR','PHONE NUMBER')
agendas_01=agendas_01.withColumnRenamed('MONTO','OFERTA_MAX_TEXTO')
agendas_01 = agendas_01.withColumnRenamed("#¡VALOR!", "TIENDA_CERCANA_LIMPIA")

In [16]:
agendas_01=agendas_01.withColumn('AGENCIA',F.lit(' '))


In [18]:
df_filtrado_2=df_filtrado_1.unionByName(agendas_01)

In [38]:
df_list_filtrada = agendas_01.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'ssssssss.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [52]:
query = """
    select *
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where (cl_telf1 is not null and cl_telf1<>0) 
    and cl_base='julio 2026'
    and OFERTA_MAX>=5000
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
print(df_formato.columns)

['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUEL

In [ ]:
['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUELDO', 'CAPACIDAD_MAX', 'PEER', 'PROP_COMER', 'TIPO_GEST', 'CLIENTE_NUEVO', 'GRUPO_TASA', 'NUEVOS_3M', 'NUEVOS_6M', 'NUEVOS_9M', 'NUEVOS_12M', 'NUEVOS_4M', 'GRUPO_MONTO', 'TASA_VS_MONTO', 'USUARIO', 'incremento_monto_riesgos', 'FLG_DEUDA_PLUS', 'tipo_cliente_riegos', 'USER_V3', 'LEAD_CALIDAD', 'SEGMENTO_USER', 'RANGO_EDAD', 'RANGO_OFERTA2', 'PERIODO', 'RETIRO_GEST', 'MEJOR_TIPIFICACION', 'STATUS', 'FECHA_SOL', 'BASE', 'RESULTADO', 'NUM_ENRIQUECIDO', 'TIPO_CONTACTO', 'Q_VENTAS', 'LOCALIDAD', 'DESEMBOLSADO', 'MONTO_DESEMBOLSADO', 'SBI', 'CRUCE', 'PREST_PREVIO', 'ID_CLIENTE', 'RANGO_EDAD2', 'Fecha_Envio', 'TIPO_BD', 'COD_BD', 'NOMB_BD', 'MES_GESTION', 'TIPO_CLIENTE', 'GRUPO_TASA_REENGANCHE', 'SALDO_DIFERENCIAL_REENG', 'FLAG_REENG', 'RETIRO_DESEMBOLSO', 'FRESCURA', 'flag_deuda_v_oferta', 'MGNEG', 'PERFIL_RO', 'TIPO_BASE', 'cl_telf1', 'cl_telf2', 'cl_telf3', 'cl_telf4', 'cl_telf5', 'cl_telf6', 'cl_telf7', 'cl_telf8', 'cl_telf9', 'cl_telf10', 'cl_movil', 'cl_celular', 'cl_telefono', 'cl_turno', 'cl_gestor', 'cl_asesor', 'cl_accion', 'cl_gestion', 'cl_estado', 'cl_fecha_gestion', 'cl_hora_gestion', 'cl_hits', 'cl_fecha_llamar', 'cl_prioridad', 'cl_orden', 'cl_predictivo', 'cl_tiempo', 'cl_base', 'cl_mes', 'cl_carga', 'id_carga', 'cl_area', 'fecha_alimentacion', 'cl_base_ant', 'cl_accion_ant', 'cl_fecha_ant', 'campania', 'PROMOCION', 'PROMOCION2', 'nombre_base', 'NumEntidades', 'p_banco', 'PERFIL_GLOBAL', 'FLG_AAHH', 'SCORE_TELEFONO', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'marca1', 'marca2', 'marca3', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE']

In [75]:
df_filtrado_2.columns

['VENDOR LEAD CODE',
 'PHONE NUMBER',
 'TIENDA_CERCANA_LIMPIA',
 'AGENCIA',
 'TIENDA_2_DIRECCION',
 'OFERTA_MAX_TEXTO',
 'ADDRESS1',
 'campana']

In [78]:
df_filtrado_2.columns

['VENDOR LEAD CODE',
 'PHONE NUMBER',
 'TIENDA_CERCANA_LIMPIA',
 'AGENCIA',
 'TIENDA_2_DIRECCION',
 'OFERTA_MAX_TEXTO',
 'ADDRESS1',
 'campana']

In [19]:
query = """
SELECT agencia_Formulario as AGENCIA1 ,agencia_correo as TIENDA_CERCANA_LIMPIA
FROM OPENQUERY([192.168.2.100], '
    select * from Alice.agencias_alfin
')
    """
df_Age=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [20]:
df_filtrado_2=df_filtrado_2.join(df_Age,['TIENDA_CERCANA_LIMPIA'],'left')

In [21]:
df_filtrado_2=df_filtrado_2.withColumn('AGENCIA',when(F.col('AGENCIA').isNull(),F.col('AGENCIA1')).otherwise(F.col('AGENCIA')))
df_filtrado_2=df_filtrado_2.withColumn('AGENCIA',when(F.col('AGENCIA')=='',F.col('AGENCIA1')).otherwise(F.col('AGENCIA')))

In [22]:
df_filtrado_2=df_filtrado_2.drop('AGENCIA1')

In [23]:

df_filtrado_2 = df_filtrado_2.withColumn(
    "AGENCIA",
    F.regexp_extract(F.col("AGENCIA"), r"^(\d+)", 1)
)

In [24]:
df_filtrado_2.show()

+---------------------+----------------+------------+-------+------------------+----------------+--------------------+-------+
|TIENDA_CERCANA_LIMPIA|VENDOR LEAD CODE|PHONE NUMBER|AGENCIA|TIENDA_2_DIRECCION|OFERTA_MAX_TEXTO|            ADDRESS1|campana|
+---------------------+----------------+------------+-------+------------------+----------------+--------------------+-------+
|               CAÑETE|        02848126|   967948335| 732243|    NARANJA OSCURO|            5200|PROFETA PEÃ‘A BERMEO|    401|
|              SULLANA|        02838605|   943747319| 734270|      VERDE OSCURO|           11100| EMERITA NIÃ‘O NIÃ‘O|    401|
|               HUACHO|        02839950|   904866097| 739467|      VERDE OSCURO|            6000|JAVIER EDUARDO PO...|    401|
|             CASTILLA|        02840567|   939146665| 737490|      VERDE OSCURO|           18000|WILFREDO ABDUL CA...|    401|
|             CASTILLA|        02707992|   920598982| 737490|      VERDE OSCURO|           11600|     JOSE MEJI

In [25]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy('TIENDA_CERCANA_LIMPIA','OFERTA_MAX_TEXTO').orderBy(F.rand())

df_split = df_filtrado_2.withColumn("grupo_split", F.ntile(3).over(w))

df_parte_1 = df_split.filter(F.col("grupo_split") == 1).drop("grupo_split")
df_parte_2 = df_split.filter(F.col("grupo_split") == 2).drop("grupo_split")
df_parte_3 = df_split.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_3 = df_split.filter(F.col("grupo_split") == 3).drop("grupo_split")
# df_parte_4 = df_split.filter(F.col("grupo_split") == 4).drop("grupo_split")
# df_parte_5 = df_split.filter(F.col("grupo_split") == 5).drop("grupo_split")
# df_parte_6 = df_split.filter(F.col("grupo_split") == 6).drop("grupo_split")
# df_parte_7 = df_split.filter(F.col("grupo_split") == 7).drop("grupo_split")

print(
    df_parte_1.count(),
    df_parte_2.count(),
    df_parte_3.count()
    # df_parte_4.count(),
    # df_parte_5.count(),
    # df_parte_6.count(),
    # df_parte_7.count()
)

9993 6895 5130


In [26]:
df_list_filtrada = df_parte_1.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin1.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)
df_list_filtrada = df_parte_2.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin2.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)
df_list_filtrada = df_parte_3.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin3.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [97]:
df_parte_3.show()

+---------------------+----------------+------------+-------+------------------+----------------+--------------------+-------+
|TIENDA_CERCANA_LIMPIA|VENDOR LEAD CODE|PHONE NUMBER|AGENCIA|TIENDA_2_DIRECCION|OFERTA_MAX_TEXTO|            ADDRESS1|campana|
+---------------------+----------------+------------+-------+------------------+----------------+--------------------+-------+
|           AREQ CAYMA|        29659831|   977319165|       |      VERDE OSCURO|           13600|FLOR LUZ VILLEGAS...|    401|
|           AREQ CAYMA|        45640479|   959155400|       |    AMARILLO CLARO|           16000|Alexander pinto p...|    401|
|           AREQ CAYMA|        29647727|   934562868|       |      VERDE OSCURO|           16000|SEBASTIANA CRISTI...|    401|
|           AREQ CAYMA|        46725183|   915384977|       |      VERDE OSCURO|           16000|ANACAREN JULIANA ...|    401|
|           AREQ CAYMA|        29472104|   949483947|       |      VERDE OSCURO|           20000|JULIA CONDORI 

In [ ]:
# append_table_SQL(spark,df_lista,f'suuuuuuubirborrddar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# 

In [2]:
query = """
    select *
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where (cl_telf1 is not null and cl_telf1<>0) 
    and cl_base='julio 2026'
    and OFERTA_MAX>=5000
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

# df_formato=df_formato.withColumnRenamed('NUMERO_DOCUMENTO','VENDOR LEAD CODE')
# df_formato=df_formato.withColumnRenamed('cl_telf1','PHONE NUMBER')
# df_formato=df_formato.withColumnRenamed('Agencia_comercial','TIENDA_CERCANA_LIMPIA')
# df_formato=df_formato.withColumn('campana',F.lit('401'))
# df_formato=df_formato.withColumnRenamed('COLOR_FINAL','TIENDA_2_DIRECCION')
# df_formato=df_formato.withColumnRenamed('OFERTA_MAX','OFERTA_MAX_TEXTO')
# df_formato=df_formato.withColumnRenamed('MARCA1','AGENCIA')
# df_formato = df_formato.withColumn(
#     "ADDRESS1",
#     F.concat_ws(
#         " ",
#         F.col("NOMBRES"),
#         F.col("APELLIDO_PATERNO"),
#         F.col("APELLIDO_MATERNO")
#     )
# )


In [ ]:
|TIPO_DOI|VENDOR LEAD CODE|         NOMBRES|APELLIDO_PATERNO|  APELLIDO_MATERNO|SUCURSAL|  TIENDA|DEPARTAMENTO|       PROVINCIA|   DISTRITO|FEC_NACIMIENTO|OFERTA_MAX_TEXTO|OFERTA_REEN|Tipo_verificacion|GRUPO_RIESGO|proveedor|          lote|estado|Tasa_1|Tasa_2|Tasa_3|Tasa_4|Tasa_5|Tasa_6|Tasa_7|segmento|campana|PLAZO| TEM|PROPENSION_IC|Desgravamen|  CUOTA|Edad|Oferta_12M|Tasa_12M|Desgravamen_12M|CUOTA_12M|Oferta_18M|Tasa_18M|Desgravamen_18M|CUOTA_18M|Oferta_24M|Tasa_24M|Desgravamen_24M|CUOTA_24M|Oferta_36M|Tasa_36M|Desgravamen_36M|CUOTA_36M|Validador_Telefono|Prioridad|Nombre_prioridad|Deuda_1|Entidad_1|Deuda_2|Entidad_2|Deuda_3|Entidad_3|sucursal_comercial|TIENDA_CERCANA_LIMPIA|    Region_comercial|Ubicacion|OfertaMaximaSinSeguro|color|TIENDA_2_DIRECCION|PROPENSION|OFERTA_FINAL|GARANTIA|Oferta_Minima_Paperless|RANGO_OFERTA|RANGO_SUELDO|CAPACIDAD_MAX|PEER|PROP_COMER|TIPO_GEST|CLIENTE_NUEVO|   GRUPO_TASA|NUEVOS_3M|NUEVOS_6M|NUEVOS_9M|NUEVOS_12M|NUEVOS_4M|    GRUPO_MONTO|TASA_VS_MONTO|USUARIO|incremento_monto_riesgos|FLG_DEUDA_PLUS|tipo_cliente_riegos|             USER_V3|LEAD_CALIDAD|SEGMENTO_USER|RANGO_EDAD|RANGO_OFERTA2|PERIODO|RETIRO_GEST|MEJOR_TIPIFICACION|STATUS|FECHA_SOL|BASE|RESULTADO|NUM_ENRIQUECIDO|TIPO_CONTACTO|Q_VENTAS|LOCALIDAD|DESEMBOLSADO|MONTO_DESEMBOLSADO| SBI|CRUCE|PREST_PREVIO|ID_CLIENTE|RANGO_EDAD2|Fecha_Envio|TIPO_BD|COD_BD|NOMB_BD|MES_GESTION| TIPO_CLIENTE|GRUPO_TASA_REENGANCHE|SALDO_DIFERENCIAL_REENG|FLAG_REENG|RETIRO_DESEMBOLSO|FRESCURA|flag_deuda_v_oferta|MGNEG|PERFIL_RO|       TIPO_BASE|PHONE NUMBER| cl_telf2|cl_telf3|cl_telf4|cl_telf5|cl_telf6|cl_telf7|cl_telf8|cl_telf9|cl_telf10|cl_movil|cl_celular|cl_telefono|cl_turno|cl_gestor|cl_asesor|cl_accion|cl_gestion|cl_estado|cl_fecha_gestion|cl_hora_gestion|cl_hits|cl_fecha_llamar|cl_prioridad|cl_orden|cl_predictivo|cl_tiempo|   cl_base|cl_mes|  cl_carga|id_carga|cl_area|fecha_alimentacion|cl_base_ant|cl_accion_ant|cl_fecha_ant|            campania|PROMOCION|PROMOCION2|nombre_base|NumEntidades|p_banco|PERFIL_GLOBAL|    FLG_AAHH|SCORE_TELEFONO|PILOTO_PLAZAS|INTENSIDAD_MAX|          AGENCIA|marca2|marca3|AÑO_DURACION_BASE|MES_DURACION_BASE|            ADDRESS1|


In [97]:
df_formato.show()

+--------+----------------+----------------+----------------+------------------+--------+--------+------------+----------------+-----------+--------------+----------+-----------+-----------------+------------+---------+--------------+------+------+------+------+------+------+------+------+--------+-------+-----+----+-------------+-----------+-------+----+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+----------+--------+---------------+---------+------------------+---------+----------------+-------+---------+-------+---------+-------+---------+------------------+-----------------+--------------------+---------+---------------------+-----+---------------+----------+------------+--------+-----------------------+------------+------------+-------------+----+----------+---------+-------------+-------------+---------+---------+---------+----------+---------+---------------+-------------+-------+-------

In [ ]:
df_prueba_1 = df_prueba_1.withColumn(
    "RANGO_OFERTA",   
    F.when(F.col("OFERTA").cast("int").isNull(), "SIN DATO")
     .when(F.col("OFERTA").cast("int") < 2500,  "00.[0 - 2,500)")
     .when(F.col("OFERTA").cast("int") < 5000,  "01.[2,500 - 5,000)")
     .when(F.col("OFERTA").cast("int") < 7500,  "02.[5,000 - 7,500)")
     .when(F.col("OFERTA").cast("int") < 10000, "03.[7,500 - 10,000)")
     .when(F.col("OFERTA").cast("int") < 12500, "04.[10,000 - 12,500)")
     .when(F.col("OFERTA").cast("int") < 15000, "05.[12,500 - 15,000)")
     .when(F.col("OFERTA").cast("int") < 17500, "06.[15,000 - 17,500)")
     .when(F.col("OFERTA").cast("int") < 20000, "07.[17,500 - 20,000)")
     .when(F.col("OFERTA").cast("int") < 22500, "08.[20,000 - 22,500)")
     .when(F.col("OFERTA").cast("int") < 25000, "09.[22,500 - 25,000)")
     .when(F.col("OFERTA").cast("int") < 27500, "10.[25,000 - 27,500)")
     .otherwise("11.[27,500 A MÁS]")
)

In [ ]:
USER_V3=['3. MES + PLD Peers', '1. sunedu & sunarp A', '12. Independiente A', '8. Tarjetero Cash', '5. MES A', '2. sunedu & sunarp B', '7. Peers', '13. No dependiente + Convenios', '6. MES B', '11. Dependiente Banca', '10. Dependiente + Convenios', '4. MES + PLD No Peers', '9. Dependiente BN', '14. Otros Bancarizados']
campania=['MUJER', 'BANTRAD', 'SDFCP100', 'SOLO CON DNI - RECUPERO', 'SD PE', 'SD FCP', 'SOLO CON DNI - ZONA COBERTURA', 'SOLODNI']
TIPO_BASE=['Blacklist', 'Regular', 'SALDO COMPETIDOR']
PROPENSION_IC=['3', '5', '6', '1', '4', '2']
Edad=[31, 65, 53, 78, 34, 28, 76, 44, 47, 52, 40, 57, 54, 48, 64, 41, 43, 37, 61, 72, 35, 59, 55, 39, 49, 51, 69, 63, 77, 50, 45, 38, 73, 70, 62, 29, 60, 32, 75, 56, 58, 33, 71, 68, 42, 79, 30, 66, 67, 46, 74, 36]
GRUPO_TASA=['Mantiene tasa', 'Mayor tasa', 'Menor tasa', None]
GRUPO_MONTO=['Mantiene Monto', 'Disminuye Monto', 'Mejora Monto', None]
color_final=['NARANJA CLARO', 'AMARILLO CLARO', 'NARANJA OSCURO', 'AMARILLO OSCURO', 'VERDE OSCURO', 'VERDE CLARO']
Region_comercial=['REGION SUR', 'REGION LIMA CENTRO', 'REGION LIMA SUR ORIENTE', 'REGION LIMA NORTE', 'REGION NORTE', None]
[74.5, 67.0, 70.0, 69.0, 59.5, 76.5, 73.5, 67.5, 52.5, 88.0, 49.0, 66.5, 98.0, 75.0, 64.0, 47.0, 81.5, 62.0, 96.0, 80.0, 86.0, 64.5, 94.0, 68.5, 85.0, 77.5, 77.0, 56.0, 50.0, 65.5, 78.0, 48.5, 79.0, 83.0, 45.0, 54.5, 57.5, 71.0, 93.0, 50.5, 58.0, 72.0, 51.0, 63.0, 48.0, 82.0, 60.0, 74.0, 66.0, 75.5, 68.0, 53.0, 61.0, 53.5, 59.0, 81.0, 78.5, 46.0, 57.0, 87.0, 60.5, 69.5, 73.0, 87.5, 99.0, 95.0, 102.0, 84.0, 92.0, 52.0, 97.0, 62.5, 89.0, 55.0, 70.5, 55.5, 84.5, 65.0, 51.5, 54.0, 76.0, 90.0, 91.0]

In [ ]:
print([row['USER_V3' ] for row in df_formato.select('USER_V3').distinct().collect()])
print([row['campania' ] for row in df_formato.select('campania').distinct().collect()])
print([row['TIPO_BASE' ] for row in df_formato.select('TIPO_BASE').distinct().collect()])
print([row['PROPENSION_IC' ] for row in df_formato.select('PROPENSION_IC').distinct().collect()])
print([row['OFERTA_MAX' ] for row in df_formato.select('OFERTA_MAX').distinct().collect()])
print([row['Edad' ] for row in df_formato.select('Edad').distinct().collect()])
print([row['GRUPO_TASA' ] for row in df_formato.select('GRUPO_TASA').distinct().collect()])
print([row['GRUPO_MONTO' ] for row in df_formato.select('GRUPO_MONTO').distinct().collect()])
print([row['color_final' ] for row in df_formato.select('color_final').distinct().collect()])
print([row['Region_comercial' ] for row in df_formato.select('Region_comercial').distinct().collect()])
print([row['Tasa_1' ] for row in df_formato.select('Tasa_1').distinct().collect()])


['3. MES + PLD Peers', '1. sunedu & sunarp A', '12. Independiente A', '8. Tarjetero Cash', '5. MES A', '2. sunedu & sunarp B', '7. Peers', '13. No dependiente + Convenios', '6. MES B', '11. Dependiente Banca', '10. Dependiente + Convenios', '4. MES + PLD No Peers', '9. Dependiente BN', '14. Otros Bancarizados']
['MUJER', 'BANTRAD', 'SDFCP100', 'SOLO CON DNI - RECUPERO', 'SD PE', 'SD FCP', 'SOLO CON DNI - ZONA COBERTURA', 'SOLODNI']
['Blacklist', 'Regular', 'SALDO COMPETIDOR']
['3', '5', '6', '1', '4', '2']
[5300, 9900, 21700, 18800, 11500, 16500, 15100, 19200, 11800, 24200, 6500, 13500, 23000, 5100, 16800, 21200, 20800, 23100, 7800, 8500, 6300, 16000, 9700, 23900, 10800, 24900, 17400, 19100, 6100, 7600, 12000, 10600, 19000, 15700, 5500, 20900, 13000, 7400, 20700, 18100, 18900, 8200, 16400, 18500, 23700, 14600, 5900, 10900, 22500, 5000, 12500, 7500, 14500, 15200, 16300, 24000, 22400, 8800, 13300, 25000, 6900, 18700, 11900, 8700, 10000, 6600, 22000, 15400, 17500, 14800, 24300, 17900, 670

In [ ]:
print([row['FRESCURA' ] for row in df_formato.select('FRESCURA').distinct().collect()])

['3', '0', '1', '4', '2']


In [ ]:
df_list_filtrada = df_formato.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'credicash_1.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

In [ ]:
['MUJER', 'BANTRAD', 'SDFCP100', 'SOLO CON DNI - RECUPERO', 'SD PE', 'SD FCP', 'SOLO CON DNI - ZONA COBERTURA', 'SOLODNI']


In [ ]:
['3. MES + PLD Peers', '1. sunedu & sunarp A', '12. Independiente A', '8. Tarjetero Cash', '5. MES A', '2. sunedu & sunarp B', '7. Peers', '13. No dependiente + Convenios', '6. MES B', '11. Dependiente Banca', '10. Dependiente + Convenios', '4. MES + PLD No Peers', '9. Dependiente BN', '14. Otros Bancarizados']


In [5]:
# grupo_tasa=['None', '', '', '', '', ]
# agencia_comercial=[ 'SAN MIGUEL', 'MIRAFLORES']

df_filtrado = df_formato.filter(
    # (F.col('propension_ic').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    # (
    (F.col('propension_ic').isin('1'))&
    (F.col('frescura').isin('4','2'))&
    # )
    (F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','3. MES + PLD Peers'))&
    (F.col('TIPO_CLIENTE').isin('INDEPENDIENTE'))&
    (F.col('Tasa_1')!=0)

)



print(df_filtrado.count())
print(df_filtrado.columns)

5942
['TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO

In [ ]:

df_dni = df_filtrado.toPandas()

ruta_archivo = os.path.join(ruta_csv, 'df_parte_3.xlsx')

df_dni.to_excel(ruta_archivo, index=False)

In [ ]:
df_filtrado_1=df_filtrado.select('VENDOR LEAD CODE','PHONE NUMBER','TIENDA_CERCANA_LIMPIA','AGENCIA','TIENDA_2_DIRECCION','OFERTA_MAX_TEXTO','ADDRESS1','campana')
df_filtrado_1.show()

In [ ]:
df_list_filtrada = df_filtrado_1.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin_1.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

### lista agente

In [4]:
query = """
    select *
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where (cl_telf1 is not null and cl_telf1<>0) 
    and cl_base='julio 2026'
    and OFERTA_MAX>=5000
    and cl_telf1<>0
    and cl_telf1 is not null
    AND LOTE<>'BOT'
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

In [ ]:
query = """
    SELECT NUMERO_DOCUMENTO FROM maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where RECORRIDO = 0
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [6]:
df_formato=df_formato.join(df_recorrido,['NUMERO_DOCUMENTO'],'leftanti')

In [4]:
print([row['LOTE' ] for row in df_formato.select('LOTE').distinct().collect()])


['BOT', 'NO CLIENTE', 'NO CLIENTE CET']


In [11]:
# grupo_tasa=['None', '', '', '', '', ]
# agencia_comercial=[ 'SAN MIGUEL', 'MIRAFLORES']

df_filtrado = df_formato.filter(
    # (F.col('propension_ic').isin('1','2','3'))&
    # (F.col('proveedor').isin(['INVENTARIO TARGET 2', 'CET TARGET 2', 'CET TARGET 0', 'INVENTARIO TARGET 1', 'INVENTARIO TARGET 0', ]))&
    # (F.col('user_v3').isin(user_v3))&

    # (
    (F.col('propension_ic').isin('1'))&
    (F.col('frescura').isin('1','4'))&
    # )
    (F.col('USER_V3').isin('7. Peers','4. MES + PLD No Peers','3. MES + PLD Peers'))&
    # (F.col('TIPO_CLIENTE').isin('INDEPENDIENTE'))&
    (F.col('Tasa_1')!=0)

)



print(df_filtrado.count())
print(df_filtrado.columns)

4392
['NUMERO_DOCUMENTO', 'TIPO_DOI', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO

In [14]:
print(df_filtrado.columns)


['NUMERO_DOCUMENTO', 'TIPO_DOI', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'proveedor', 'lote', 'RETIRO', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', 'RANGO_OFERTA', 'RANGO_SUEL

In [7]:

df_filtrado=df_filtrado.withColumnRenamed('NUMERO_DOCUMENTO','vendor_lead_code')
df_filtrado=df_filtrado.withColumnRenamed('cl_telf1','phone_number')
# df_filtrado=df_filtrado.withColumnRenamed('NOMBRES','address1')

df_filtrado = df_filtrado.withColumn(
    "address1",
    F.concat_ws(
        " ",
        F.col("NOMBRES"),
        F.col("APELLIDO_PATERNO"),
        F.col("APELLIDO_MATERNO")
    )
)

df_filtrado = df_filtrado.withColumn(
    "security_phrase",
    F.concat_ws(
        " ",
        F.lit("Oferta:"),
        F.col("OFERTA_MAX")
    )
)
df_filtrado = df_filtrado.withColumn(
    "city",
    F.concat_ws(
        " ",
        F.lit("Departamento:"),
        F.col("DEPARTAMENTO")
    )
)
df_filtrado = df_filtrado.withColumn(
    "province",
    F.concat_ws(
        " ",
        F.lit("Distrito:"),
        F.col("DISTRITO")
    )
)

df_filtrado.count()

5942

In [8]:
df_filtrado=df_filtrado.select('vendor_lead_code','phone_number','address1','security_phrase','city','province')

In [9]:
df_filtrado=df_filtrado.dropDuplicates(['vendor_lead_code'])

In [10]:
df_filtrado.count()

5942

In [11]:
df_list_filtrada = df_filtrado.toPandas()
ruta_archivo = os.path.join(ruta_csv, 'alfin.xlsx')
df_list_filtrada.to_excel(ruta_archivo, index=False)

### simular

In [50]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [ ]:
query = """
    select NUMERO_DOCUMENTO as Dni,cl_telf1 as PHONE_NUMBER
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where cl_base='julio 2026'
    and cl_telf1<>0
    and cl_telf1 is not null
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)
print(df_formato.columns)




['Dni', 'PHONE_NUMBER']


In [55]:
query = """
    select NUMERO_DOCUMENTO as Dni,PROPENSION_IC,FRESCURA,USER_V3,TIPO_LOTE,lote
    from maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where retiro='Activo'
    and recorrido=1
    """
df_ref_1=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_ref.count())

129832


In [56]:
df_ref_1=df_ref_1.join(df_formato,['Dni'],'inner')
df_ref_1.count()

129809

In [71]:
df_ref=df_ref_1.filter(
        (
            F.col('lote')=='NO CLIENTE')&(F.col('USER_V3').isin('7. Peers'))&(F.col('PROPENSION_IC').isin(1))&(F.col('frescura').isin(1)
        )|
        (
           ( F.col('lote')=='BOT')&(F.col('PROPENSION_IC').isin(3,2,5,4,1))&(F.col('frescura').isin(1,2,3,4,5))
        )
    )
df_ref.count()

33719

In [72]:
df_ref.show()

+--------+-------------+--------+--------------------+------------+----------+------------+
|     Dni|PROPENSION_IC|FRESCURA|             USER_V3|   TIPO_LOTE|      lote|PHONE_NUMBER|
+--------+-------------+--------+--------------------+------------+----------+------------+
|01336535|            5|       2|14. Otros Bancari...|    BASE BOT|       BOT|   980703559|
|01225454|            2|       1|2. sunedu & sunarp B|    BASE BOT|       BOT|   978596492|
|03238433|            4|       3|14. Otros Bancari...|    BASE BOT|       BOT|   941799996|
|07148138|            3|       4|14. Otros Bancari...|    BASE BOT|       BOT|   992308745|
|04428228|            4|       4|14. Otros Bancari...|    BASE BOT|       BOT|   959131814|
|04435537|            5|       2|2. sunedu & sunarp B|    BASE BOT|       BOT|   998557787|
|06157500|            5|       4|2. sunedu & sunarp B|    BASE BOT|       BOT|   990978522|
|04748027|            2|       2|14. Otros Bancari...|    BASE BOT|       BOT|  

In [14]:
df_ref.groupBy('USER_V3') \
    .count() \
    .orderBy('USER_V3') \
    .show(30,truncate=False)


+------------------------------+-----+
|USER_V3                       |count|
+------------------------------+-----+
|1. sunedu & sunarp A          |1684 |
|10. Dependiente + Convenios   |35   |
|11. Dependiente Banca         |74   |
|12. Independiente A           |23   |
|13. No dependiente + Convenios|22   |
|14. Otros Bancarizados        |37128|
|2. sunedu & sunarp B          |13431|
|3. MES + PLD Peers            |1030 |
|4. MES + PLD No Peers         |287  |
|5. MES A                      |741  |
|6. MES B                      |19882|
|7. Peers                      |2962 |
|8. Tarjetero Cash             |168  |
|9. Dependiente BN             |73   |
+------------------------------+-----+



In [76]:
query = """
SELECT *
FROM OPENQUERY([192.168.2.100], '
    select *,dni_cliente as Dni from Alice.prospectos_correos_alfin
    WHERE fecha_dia IN (''2026-07-10'', ''2026-07-01'')
')
    """
df_crm=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_crm=df_crm.select( 'Dni', 'fecha_visita', 'hora_visita', 'fecha_envio', 'intentos_realizados', 'fecha_registro','estado')

df_crm=df_crm.withColumn('ref',when(F.col('estado')=='ENVIADO',1).otherwise(2))

window_spec = Window.partitionBy("Dni").orderBy(col("ref").asc_nulls_last())
df_crm = df_crm.withColumn("n_mejor_resul_dia", row_number().over(window_spec))
df_crm=df_crm.filter((F.col('n_mejor_resul_dia')==1)&(F.col('ref')==1)).drop('ref','n_mejor_resul_dia')

In [77]:
# df_crm=df_crm.withColumnRenamed('Dni','DNI')
df_crm = df_crm.withColumn(
    "Dni",
    F.right(
        F.concat(F.lit("00000000"), F.col("Dni")),
        F.lit(8)
    )

)


In [78]:
df_ref=df_ref.join(df_crm,['Dni'],'left')

In [80]:
w = Window.partitionBy('lote','PROPENSION_IC','FRESCURA','USER_V3').orderBy(F.rand())

df_split = df_ref.withColumn("por_dia", F.ntile(2).over(w))

In [82]:
df_split=df_split.withColumn('por_dia',when(F.col('por_dia')==2,10).otherwise(F.col('por_dia')))

In [83]:
df_split.groupBy('por_dia') \
    .count() \
    .orderBy('por_dia') \
    .show(30)


+-------+-----+
|por_dia|count|
+-------+-----+
|      1|16883|
|     10|16836|
+-------+-----+



In [10]:

df_split = df_split.withColumn(
    "por_dia",
    F.when(F.to_date("fecha_registro") == F.lit("2026-07-08"), 6)
     .when(F.to_date("fecha_registro") == F.lit("2026-07-09"), 7)
     .otherwise(F.col("por_dia"))
)

In [84]:
df_split=df_split.withColumn('Numero_Campana',F.lit('401'))
df_split=df_split.withColumn('list_description',F.lit('provicional'))
df_split=df_split.withColumn('list_name',F.lit('provicional'))
df_split=df_split.withColumn('Nombre_Campana',when(F.lit('lote')=='BOT',F.lit('BOT_ALFIN')).otherwise(F.lit('2026-07 BCO ALFIN')))


In [12]:
df_split=df_split.withColumn('por_dia',when(F.col('por_dia')==5,6)
.when(F.col('por_dia')==6,7)
.when(F.col('por_dia')==7,8).otherwise(F.col('por_dia')))

In [85]:
df_split = df_split.withColumn(
    "Fecha_Llamada",
    F.to_date(
        F.concat(
            F.lit("2026-07-"),
            F.lpad(F.col("por_dia").cast("string"), 2, "0")
        )
    )
)

In [14]:

query = """
select top(10)* From SAMANTHA.dbo.tmp_llamadas_mes
    """
df_necesito=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_necesito=df_necesito.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description')

In [15]:
query = """
select top(10)* From SAMANTHA.dbo.tmp_llamadas_mes
    """
df_necesito=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_necesito.show(4)

+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|     Dni|Numero_Campana|      Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|Descripcion|    list_description|           list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin|lead_id|
+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|72048296|            80|2023-11 CENCOSUD ...|         VDAD|Outbound Auto Dial|

In [86]:
usuarios_regular = [
    "PEC200",
    "PEC188",
    "PEC136",
    "PEC139",
    "PEC184",
    'VDAD'
]
usuarios_bot = [
    "49587612",
    'VDAD'
]

In [87]:

# Probabilidad base de VDAD por TIPO_LOTE
df_split = df_split.withColumn(
    "prob_vdad_base",
    F.when(F.col("TIPO_LOTE") == "BASE REGULAR", F.lit(0.43))
     .when(F.col("TIPO_LOTE") == "BASE BOT", F.lit(0.96))
     .otherwise(F.lit(0.43))
)

# Score: mientras más alto PROPENSION_IC y FRESCURA, más chance de VDAD
df_split = df_split.withColumn(
    "score_vdad",
    (
        (F.col("PROPENSION_IC") - 1) / F.lit(5) +   # PROPENSION_IC 1 a 6
        (F.col("FRESCURA") / F.lit(5))           # FRESCURA 0 a 5
    ) / F.lit(2)
)

# Ajustamos la probabilidad de VDAD
# Ejemplo: puede subir hasta +15 puntos porcentuales
df_split = df_split.withColumn(
    "prob_vdad_final",
    F.least(
        F.col("prob_vdad_base") + (F.col("score_vdad") * F.lit(0.15)),
        F.lit(0.99)
    )
)

# Número aleatorio
df_split = df_split.withColumn("rnd", F.rand())

# Asignar primero VDAD según probabilidad
df_split = df_split.withColumn(
    "DNI_Ejecutivo",
    F.when(F.col("rnd") <= F.col("prob_vdad_final"), F.lit("VDAD"))
)

# Para los demás, asignación uniforme según TIPO_LOTE
regular_sin_vdad = [x for x in usuarios_regular if x != "VDAD"]
bot_sin_vdad = [x for x in usuarios_bot if x != "VDAD"]

df_split = df_split.withColumn(
    "DNI_Ejecutivo",
    F.when(
        F.col("DNI_Ejecutivo").isNull() & (F.col("TIPO_LOTE") == "BASE REGULAR"),
        F.array(*[F.lit(x) for x in regular_sin_vdad])[
            F.floor(F.rand() * F.lit(len(regular_sin_vdad))).cast("int")
        ]
    )
    .when(
        F.col("DNI_Ejecutivo").isNull() & (F.col("TIPO_LOTE") == "BASE BOT"),
        F.array(*[F.lit(x) for x in bot_sin_vdad])[
            F.floor(F.rand() * F.lit(len(bot_sin_vdad))).cast("int")
        ]
    )
    .otherwise(F.col("DNI_Ejecutivo"))
)

df_split = df_split.drop("prob_vdad_base", "score_vdad", "prob_vdad_final", "rnd")

In [88]:
df_split = df_split.withColumn(
    "Ejecutivo",
    F.when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") == "VDAD"),
        F.lit("Outbound Auto Dial")
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") != "VDAD"),
        F.lit("Bot Bco Alfin")
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR") & 
        (F.col("DNI_Ejecutivo").isin([u for u in usuarios_regular if u != "VDAD"])),
        F.lit("Agente BCO Alfin")
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR") & (F.col("DNI_Ejecutivo") == "VDAD"),
        F.lit("Outbound Auto Dial")
    )
    .otherwise(F.lit(None))
)

In [ ]:
codigos_vdad = ["PDROP", "NA", "AB"]

codigos_regular = ["zzz29", "zzz32", "zzz30", "zzz34", "zzz10", "zzz56", "zzz5", "zzz13", "zzz10", "zzz9"]

codigos_bot = ["zzz29", "zzz32", "zzz30", "zzz34", "zzz10", "zzz9", "zzz56", "zzz13"]

arr_vdad = F.array(*[F.lit(x) for x in codigos_vdad])
arr_regular = F.array(*[F.lit(x) for x in codigos_regular])
arr_bot = F.array(*[F.lit(x) for x in codigos_bot])

df_split = df_split.withColumn(
    "Codigo_Paleta",
    F.when(
        F.col("DNI_Ejecutivo") == "VDAD",
        arr_vdad[F.floor(F.rand() * F.lit(len(codigos_vdad))).cast("int")]
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE REGULAR") & (F.col("DNI_Ejecutivo") != "VDAD"),
        arr_regular[F.floor(F.rand() * F.lit(len(codigos_regular))).cast("int")]
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") != "VDAD"),
        arr_bot[F.floor(F.rand() * F.lit(len(codigos_bot))).cast("int")]
    )
    .otherwise(F.lit(None))
)

In [92]:
df_split=df_split.withColumn('Codigo_Paleta',when(F.col('estado').isNotNull(),'zzz2').otherwise(F.col('Codigo_Paleta')))

In [93]:
df_split = df_split.withColumn(
    "segundos",
    F.when(
        F.col("Codigo_Paleta").isin("PDROP", "AB", "NA", "zzz50"),
        F.lit(0)
    )
    .when(
        F.col("Codigo_Paleta") == "zzz2",
        F.floor(F.rand() * 51 + 100).cast("int")   # 100 a 150
    )
    .when(
        F.col("Codigo_Paleta").isin("zzz29", "zzz30", "zzz34", "zzz9", "zzz32", "zzz56", "zzz10", "zzz13"),
        F.floor(F.rand() * 51 + 30).cast("int")    # 30 a 80
    )
    .when(
        F.col("Codigo_Paleta").isin("zzz29", "zzz30", "zzz34", "zzz9", "zzz32", "zzz56", "zzz10", "zzz13","zzz5", "zzz13", "zzz10", "zzz9"),
        F.floor(F.rand() * 51 + 30).cast("int")    # 30 a 80
    )
    .otherwise(F.lit(0))
)

## mas atento

In [94]:
from pyspark.sql.window import Window

fecha_col = "Fecha_Llamada"  # cambia esto si tu columna se llama fecha_envio, Fecha_Llamada, etc.

inicio = 9 * 3600          # 09:00:00
fin = 17 * 3600 + 50 * 60  # 17:50:00
rango = fin - inicio

df_split = df_split.withColumn("_id", F.monotonically_increasing_id())

# Duración de llamada
df_split = df_split.withColumn(
    "_duracion",
    F.when(F.col("segundos").isNull(), F.lit(0))
     .otherwise(F.col("segundos").cast("int"))
)

# =========================
# 1. VDAD: horario libre
# =========================
df_vdad = df_split.filter(F.col("DNI_Ejecutivo") == "VDAD") \
    .withColumn(
        "_seg_inicio",
        F.floor(F.rand() * rango + inicio).cast("int")
    )

# =========================
# 2. BASE REGULAR: sin cruce por agente y fecha
# =========================
df_regular = df_split.filter(
    (F.col("TIPO_LOTE") == "BASE REGULAR") &
    (F.col("DNI_Ejecutivo") != "VDAD")
)

w_reg = Window.partitionBy(fecha_col, "DNI_Ejecutivo").orderBy(F.rand())

df_regular = df_regular.withColumn("_rn", F.row_number().over(w_reg))

# pequeño espacio aleatorio entre llamadas: 5 a 20 segundos
df_regular = df_regular.withColumn(
    "_gap",
    F.floor(F.rand() * 16 + 5).cast("int")
)

w_reg_acum = Window.partitionBy(fecha_col, "DNI_Ejecutivo") \
    .orderBy("_rn") \
    .rowsBetween(Window.unboundedPreceding, -1)

df_regular = df_regular.withColumn(
    "_seg_inicio",
    inicio + F.coalesce(
        F.sum(F.col("_duracion") + F.col("_gap")).over(w_reg_acum),
        F.lit(0)
    )
)

# =========================
# 3. BASE BOT: hasta 15 llamadas al mismo tiempo
# =========================
df_bot = df_split.filter(
    (F.col("TIPO_LOTE") == "BASE REGULAR") &
    (F.col("DNI_Ejecutivo") != "VDAD")
)

w_bot = Window.partitionBy(fecha_col, "DNI_Ejecutivo").orderBy(F.rand())

df_bot = df_bot.withColumn("_rn", F.row_number().over(w_bot))

# cada grupo de 15 llamadas comparte la misma hora
df_bot = df_bot.withColumn(
    "_grupo_bot",
    F.floor((F.col("_rn") - 1) / 15).cast("int")
)

df_bot_grupos = df_bot.groupBy(fecha_col, "DNI_Ejecutivo", "_grupo_bot") \
    .agg(F.max("_duracion").alias("_duracion_grupo"))

df_bot_grupos = df_bot_grupos.withColumn(
    "_gap_grupo",
    F.floor(F.rand() * 16 + 5).cast("int")
)

w_bot_acum = Window.partitionBy(fecha_col, "DNI_Ejecutivo") \
    .orderBy("_grupo_bot") \
    .rowsBetween(Window.unboundedPreceding, -1)

df_bot_grupos = df_bot_grupos.withColumn(
    "_seg_inicio",
    inicio + F.coalesce(
        F.sum(F.col("_duracion_grupo") + F.col("_gap_grupo")).over(w_bot_acum),
        F.lit(0)
    )
)


In [95]:

df_bot = df_bot.join(
    df_bot_grupos.select(fecha_col, "DNI_Ejecutivo", "_grupo_bot", "_seg_inicio"),
    on=[fecha_col, "DNI_Ejecutivo", "_grupo_bot"],
    how="left"
)

In [96]:


# =========================
# Unir todo
# =========================
df_split = df_regular.unionByName(df_bot, allowMissingColumns=True) \
                     .unionByName(df_vdad, allowMissingColumns=True)

# Si se pasa de 17:50, lo limitamos a 17:50
df_split = df_split.withColumn(
    "_seg_inicio",
    F.when(F.col("_seg_inicio") > fin, F.lit(fin))
     .otherwise(F.col("_seg_inicio").cast("int"))
)

# Crear hora formato HH:mm:ss
df_split = df_split.withColumn(
    "hora",
    F.date_format(
        F.from_unixtime(F.col("_seg_inicio")),
        "HH:mm:ss"
    )
)

# Limpiar columnas auxiliares
df_split = df_split.drop(
    "_id", "_duracion", "_rn", "_gap",
    "_grupo_bot", "_seg_inicio",
    "_duracion_grupo", "_gap_grupo"
)

In [97]:
from pyspark.sql import functions as F

df_split = df_split.withColumn(
    "Fecha_Hora_Llamada",
    F.to_timestamp(
        F.concat_ws(
            " ",
            F.date_format(F.col("Fecha_Llamada"), "yyyy-MM-dd"),
            F.col("hora")
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
)

df_split = df_split.withColumn(
    "Inicio",
    F.col("Fecha_Hora_Llamada")
)

df_split = df_split.withColumn(
    "Fin",
    F.from_unixtime(
        F.unix_timestamp(F.col("Inicio")) + F.col("segundos").cast("int")
    ).cast("timestamp")
)

df_split = df_split.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)

In [98]:

df_split = df_split.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)

In [99]:
df_split=df_split.dropDuplicates(['Dni'])

In [100]:
df_split = df_split.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)

In [101]:
df_split.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description').show(3)

+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|00005798|         VDAD|Outbound Auto Dial|2026-07-10 10:13:15|       0|   2026-07-10|        10|   964376735|           NA|2026-07-10 10:13:15|2026-07-10 10:13:15|           401|        10|provicional|     provicional|
|00005965|         VDAD|Outbound Auto Dial|2026-07-01 05:43:32|       0|   2026-07-01|         5|   956621221|          

In [102]:
df_split=df_split.withColumn('Sub_estado',F.lit(''))
df_split=df_split.withColumn('Estados',F.lit(''))

In [103]:
df_split=df_split.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description','Estados','Sub_estado')

In [104]:
df_split.groupBy('Fecha_Llamada') \
    .count() \
    .orderBy('Fecha_Llamada') \
    .show(30)


+-------------+-----+
|Fecha_Llamada|count|
+-------------+-----+
|   2026-07-01|16719|
|   2026-07-10|16693|
+-------------+-----+



In [106]:
append_table_SQL(spark,df_split,f'tmp_llamadas_mes_que',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')


In [210]:
print([row['estado' ] for row in df_split.select('estado').distinct().collect()])


['ENVIADO   ', None]


In [38]:
df_necesito.show(2)
# df_necesito=df_necesito.withColumn('Numero_Campana',F.lit('401'))
# df_necesito=df_necesito.withColumn('list_description',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('list_name',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('BOT_ALFIN'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('2026-07 BCO ALFIN'))


+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|72048296|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   922488467|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
|29690194|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   983941534|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
only showi

In [205]:
df_split.show(5)

+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|     Dni|PROPENSION_IC|FRESCURA|             USER_V3|TIPO_LOTE|lote|PHONE_NUMBER|fecha_visita|hora_visita|fecha_envio|intentos_realizados|fecha_registro|estado|por_dia|Numero_Campana|list_description|  list_name|   Nombre_Campana|Fecha_Llamada|DNI_Ejecutivo|         Ejecutivo|Codigo_Paleta|
+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|47319495|            1|       0|2. sunedu & sunarp B| BASE BOT| BOT|   954821826|        NULL|       NULL|       NULL|  

In [ ]:


Bot Bco Alfin



In [186]:

query = """
select top(10)* From [THOTH].dbo.Tmp_LLamadas_Alfin_bot
    """
df_llamadas_bot=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_llamadas_bot.show()

+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+-----------+-------------+-------------------+-------------------+--------+--------+--------------------+--------------------+-----+------------------+----------+------------+---+-------+
|     Dni|Numero_Campana|Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|         Descripcion|list_description|       list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin| lead_id| Estado_|         Sub_Estado_|        Descripcion_|Pesos|            Enlace|Fecha_Llam|Hora_Llamada| RH|COD_BCO|
+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+---

In [187]:
print([row['DNI_Ejecutivo' ] for row in df_llamadas_bot.select('DNI_Ejecutivo').distinct().collect()])


['49587612', 'VDAD']


In [101]:
print(41539-41668)

-129


In [ ]:
nocet 41668
1760 odo
bot 43428

In [58]:
49587612

['ACTIVO', 'RETIVO_CORREO']


In [2]:
print(86857*.50,'bot')
print((102599+2779)*.3,'hum')
print(192235)

43428.5 bot
31613.399999999998 hum
192235


In [23]:
print([row['estado' ] for row in df_crm.select('estado').distinct().collect()])
# print([row['codigo_ejecutivo_id' ] for row in df_crm.select('codigo_ejecutivo_id').distinct().collect()])


['PENDIENTE ', 'ERROR     ', 'ENVIADO   ']


In [ ]:
['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', 'tipo_cliente_riegos', 'USER_V3', 'TIPO_CLIENTE', 'FRESCURA', 'TIPO_BASE', 'campania', 'FLG_AAHH', 'INTENSIDAD_MAX', 'REGION', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_TASA', 'TIPO_LOTE', 'TIPO_TELF', 'lote', 'tMontoDesemFugas']prop

['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', '

In [15]:
query = """
    SELECT * FROM maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where RECORRIDO = 0
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.count()

156930

In [6]:
df_formato.join(df_recorrido,['NUMERO_DOCUMENTO'],'inner').count()

0

In [29]:
df_recorrido.show()

+----------------+---+----+------------+----------+----------+----------------+-----------+--------------+----------------+-----------+--------------+----------+---------+-------------+-------------+---+-------+-------+--------+------------+------------+------------+--------------+--------+-------------+----+---------+----+-----------+--------+---------+-------------+-------------+--------+--------+--------+--------+---------+-------+------+---------+---------+-------+-------------+----------+-----------+------------+----------------+----------------+-----------------+-----------------+-----------+--------+------+-----+-----+-----------+-----------+--------------------+------------------+-------------------+------------+--------------------+-------------+-------+--------------------+--------------------+---------------+-------------+---------------+-------------------+--------------------+-------------+--------+----------------+--------+------------+--------------+---------------+-----

In [31]:
df_recorrido.count()

1999

In [33]:
# print([row['FEC_NAC' ] for row in df_list.select('FEC_NAC').distinct().collect()])
df_recorrido.groupBy('PROPENSION_IC') \
    .count() \
    .orderBy('PROPENSION_IC') \
    .show(30)


+-------------+-----+
|PROPENSION_IC|count|
+-------------+-----+
|            1|  572|
|            2|  980|
|            3|  132|
|            4|  122|
|            5|  116|
|            6|   77|
+-------------+-----+



In [ ]:
df_recorrido

In [25]:
query = """
    select top(10)* From SAMANTHA..tmp_llamadas_mes
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.show(2)

+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|     Dni|Numero_Campana|      Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|Descripcion|    list_description|           list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin|lead_id|
+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|72048296|            80|2023-11 CENCOSUD ...|         VDAD|Outbound Auto Dial|

In [ ]:
RetiroDefinitivo_BlackList